# Datathon@IndoML 2026 — Track 1: Noise Event Detection

Detect noise events in real-world Indic speech and submit onset/offset timestamps.
Everything below runs inside one Kaggle GPU session and ends with `submission_track1.zip`.

## The task

Given a Vaani speech clip, predict every interval containing a non-speech noise event —
a horn, a dog, a child, a ringtone, a pressure cooker. Submission is one JSON object per clip:

```
{"clip_id": "vaani_eval_001", "events": [{"onset": 1.24, "offset": 3.81}]}
```

Detection is **class-agnostic**: no category field.

## Scoring

`Combined = Event F1 + Segment Dice`, maximum 2.0.

- **Event F1** — a prediction matches when both onset and offset fall within
  `max(0.20 x duration, 0.05)` seconds of a reference event. Matching is global closest-first.
- **Segment Dice** — 10 ms frame masks, macro-averaged **per clip**. A clip with no reference
  events and no predictions scores **1.0**.

That Dice rule drives a design choice you will see in Section 7: staying silent on a clip you
believe is clean buys a whole point, while a single false alarm there costs the same point.

## Approach

| stage | choice | why |
|---|---|---|
| encoder | **WavLM-base+**, fine-tuned | 50 Hz frames = exactly our 20 ms label grid, and it was pretrained *with* simulated noise and overlap, so background is encoded rather than suppressed |
| head | BiGRU + frame head + attention-pooled clip head | the clip head is the only gradient path for tag-only (Bronze) data |
| training | **two-stage** fine-tuning + mean teacher | fine-tuning a large pretrained encoder end-to-end from step 0 overfits and lands *below* a from-scratch CRNN |
| labels | Gold strong / Silver down-weighted / Bronze weak | the three annotation tiers carry different information and are used differently |
| post-proc | threshold, median filter, min-duration, **clip gate** | swept jointly on cached posteriors |

## Honest framing

This is **not** ATST-SED. That paper's contribution is the two-stage schedule, which we use;
its encoder is ATST-Frame, which we do not. What is implemented is the DCASE mean-teacher CRNN
recipe with a pretrained WavLM encoder in place of a from-scratch CNN. Describe it that way.

## Setup

1. Accept the terms on `huggingface.co/datasets/ARTPARK-IISc/Vaani-Noise-Event-Dataset`.
2. Kaggle -> Add-ons -> Secrets -> add `HF_TOKEN`.
3. Internet **ON**, Accelerator **GPU**.
4. Attach the Track 1 test audio (`input_data` from the Codabench Files tab) as a Kaggle Dataset.


---
## 0. Config

The single control panel for the whole notebook — every tunable knob lives here so nothing is
hard-coded deeper down. The one knob you will actually touch is `BUDGET`.

**Budget switch**
- `BUDGET = "fast"` (~3 h) vs `"full"` (~7 h). It drives the data caps (`MAX_PER_QUALITY`) and
  epoch counts near the bottom of the cell. Toggle this to trade time for quality.

**`CFG` — core audio + training dict**

| key | value | meaning |
|---|---|---|
| `sr=16000` | sample rate | WavLM expects 16 kHz |
| `hop=160` | STFT hop | 160/16000 = **10 ms** label grid |
| `clip_sec=5.0` | window length | 5 s (not 10) halves the WavLM sequence and doubles the batch — the biggest speed lever |
| `time_pool=2` | frame pooling | pools the 10 ms grid down to **20 ms**, matching WavLM's native 50 Hz output |
| `silver_weight=0.4` | loss weight | Silver-tier labels count 40% of Gold |
| `ema_decay, max_cons_w, rampup_epochs` | | mean-teacher settings (Section 6) |
| `pos_weight=4.0` | | up-weights rare noise frames so BCE doesn't under-fire |
| `mixup_prob/alpha` | | mixup augmentation strength |
| `val_frac=0.1, seed=42` | | 10% validation split, reproducibility seed |

**Encoder settings**
- `ENCODER_NAME` — the pretrained backbone (WavLM-base+).
- `DROP_LAST_LAYERS=2` — discard the top layers that overfit the pretraining task.
- `LR_ENC, LR_HEAD` — a tiny LR for the encoder, a larger one for the head.

**Budget-dependent caps**
- `MAX_PER_QUALITY` — how many clips of each tier (Gold / Silver / Bronze) to pull.
- `STAGE1_EPOCHS, STAGE2_EPOCHS, T1_BS` — epochs per stage and batch size. `full` lifts the
  caps and doubles the epochs.

**Categories, repo, and paths**
- `CATS` / `CAT2IDX` — the 7 noise categories (predicted internally; Track 1 submits class-agnostic).
- `REPO` — the HF dataset.
- `WORK`, `CKPT` — working directory and the checkpoint output location.

**Derived constants**
- `N_SAMP`, `FRAMES_100HZ`, `FRAMES_OUT`, `FRAME_SEC` — window/frame sizes used everywhere else,
  then `random`/`numpy` are seeded.

In one sentence: this cell defines **what data to load, how audio is framed, which encoder to
fine-tune, how long to train, and where everything is written** — the rest of the notebook just
consumes these constants.


In [ ]:
import os, io, json, time, math, random, zipfile
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict

# ---- budget switch: the one knob you actually touch ----
BUDGET = "fast"            # "fast" (~3 h) or "full" (~7 h)

# ---- core audio + training config ----
CFG = dict(
    sr=16000,             # WavLM expects 16 kHz
    hop=160,              # 160/16000 = 10 ms label grid
    clip_sec=5.0,         # 5 s window: halves the WavLM sequence, doubles the batch
    time_pool=2,          # pool 10 ms -> 20 ms, matching WavLM's native 50 Hz output
    n_cat=7,              # number of noise categories (channel 0 is "any noise")
    silver_weight=0.4,    # Silver-tier labels count 40% of Gold
    ema_decay=0.999,      # mean-teacher EMA
    max_cons_w=2.0,       # max consistency weight
    rampup_epochs=5,      # consistency ramp-up length
    pos_weight=4.0,       # up-weight rare noise frames in BCE
    mixup_prob=0.5,
    mixup_alpha=0.2,
    val_frac=0.1,         # 10% validation split
    weight_decay=1e-2,
    num_workers=2,
    seed=42,
)

# ---- categories (predicted internally; Track 1 submits class-agnostic) ----
CATS = [
    "animal", "vehicle_traffic", "baby_child", "singing_music",
    "phone_signal_alarm", "appliance_machine", "human_non_speech",
]
CAT2IDX = {c: i for i, c in enumerate(CATS)}

# ---- dataset + output locations ----
REPO = "ARTPARK-IISc/Vaani-Noise-Event-Dataset"
WORK = Path("/kaggle/working")
CKPT = WORK / "t1_wavlm.pt"
WORK.mkdir(parents=True, exist_ok=True)

# ---- derived constants (window / frame sizes used everywhere else) ----
N_SAMP = int(CFG["clip_sec"] * CFG["sr"])
FRAMES_100HZ = int(CFG["clip_sec"] * CFG["sr"] / CFG["hop"])
FRAMES_OUT = FRAMES_100HZ // CFG["time_pool"]
FRAME_SEC = CFG["hop"] * CFG["time_pool"] / CFG["sr"]      # seconds per output (20 ms) frame

random.seed(CFG["seed"]); np.random.seed(CFG["seed"])
print(f"BUDGET={BUDGET} | N_SAMP={N_SAMP} | FRAMES_OUT={FRAMES_OUT} | FRAME_SEC={FRAME_SEC:.4f}s")

ENCODER_NAME = "microsoft/wavlm-base-plus"
DROP_LAST_LAYERS = 2       # Schmid et al. ICASSP 2025: final layers overfit the pretrain task
LR_ENC, LR_HEAD = 5e-5, 1e-3

if BUDGET == "fast":
    MAX_PER_QUALITY = {"verified_timestamps": 6000, "unverified_timestamps": 6000,
                       "no_timestamps": 2000}
    STAGE1_EPOCHS, STAGE2_EPOCHS, T1_BS = 3, 8, 16
    TRAIN_LIMIT = 100          # smoke-run: cap training clips. Set to None for all.
else:
    MAX_PER_QUALITY = {"verified_timestamps": None, "unverified_timestamps": 20000,
                       "no_timestamps": 8000}
    STAGE1_EPOCHS, STAGE2_EPOCHS, T1_BS = 6, 16, 16
    TRAIN_LIMIT = None         # use all available training clips

# When TRAIN_LIMIT is small, decoding thousands of clips is wasteful -> cap the prep pass too.
if TRAIN_LIMIT is not None:
    _cap = TRAIN_LIMIT + 32   # a little headroom for the gold-only validation split
    MAX_PER_QUALITY = {k: (min(v, _cap) if v is not None else _cap)
                       for k, v in MAX_PER_QUALITY.items()}


---
## 1. Data — stream the dataset and decode in one pass

Keep it simple: point `load_dataset` at the repo and read rows straight through. No shard scanning,
no manual parquet picking — the same approach as the reference script
`track1_crnn_meanteacher_baseline.py`.

**What this cell does**
1. **Auth** — logs in with your `HF_TOKEN` (from Kaggle Secrets).
2. **Load** — `load_dataset(REPO, split="train", streaming=True)` opens the dataset as a stream so
   we iterate rows without downloading every shard up front, and stop as soon as the per-tier caps
   in `MAX_PER_QUALITY` are met.
3. **`decode=False`** on the `audio` column — the `datasets` Audio feature has changed decoders
   twice (mono→num_channels, then a torchcodec `AudioDecoder`). Handing back raw **bytes** that we
   decode ourselves is the only version-proof path.

**The three annotation tiers** (used later for weighting and the val split)
- **Gold** (`verified_timestamps`) — the only fully trustworthy onset/offset supervision, and the
  source for the Track 2 clean/noise banks.
- **Silver** (`unverified_timestamps`) — timestamps without agreement; used but down-weighted.
- **Bronze** (`no_timestamps`) — clip-level tags only; feeds the weak/attention head.


In [ ]:
!pip -q install datasets huggingface_hub soundfile librosa webrtcvad-wheels transformers 2>/dev/null | tail -1
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from datasets import load_dataset, Audio
from tqdm.auto import tqdm

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)

# Stream the dataset: iterate rows directly, no shard scanning or manual parquet picking.
# streaming=True means we pull data as we go and stop once the per-tier caps are met, so we
# never download shards we won't use.
raw = load_dataset(REPO, split="train", streaming=True, token=HF_TOKEN)
# Never let `datasets` decode audio: the Audio feature changed twice (mono -> num_channels,
# and decoding now needs torchcodec and returns an AudioDecoder). decode=False works on
# every version and hands back bytes we decode ourselves.
raw = raw.cast_column("audio", Audio(decode=False))
print("dataset ready (streaming, decode=False)")


### 1b. Peek at one example

A quick sanity check on the first streamed row: the column names, one clip's annotation quality and
its timestamps. An empty timestamp list on a `no_timestamps` (Bronze) row is *correct*, not a
parsing failure.


In [ ]:
ex0 = next(iter(raw))
print("columns:", list(ex0.keys()))
print("annotationQuality:", ex0.get("annotationQuality"))
print("timestamps sample:", ex0.get("NoiseSubCategoryTimeStamp"))
print("NOTE: empty timestamps on a `no_timestamps` row is CORRECT - bronze has tags only.")


---
## 2. One pass over the audio

Each clip is decoded exactly once and kept **in memory** as an int16 waveform (WavLM consumes raw
audio, and int16 keeps the RAM footprint small). No disk cache — the same in-memory `rows` list the
reference script builds.

Labels are rasterised to a 100 Hz grid as `(8, T)`: channel 0 is "any noise event" (the
detection target), channels 1-7 are the seven Vaani categories.

Tier handling:

- **Gold** (`verified_timestamps`) — verified onset/offset, the only fully trustworthy supervision
- **Silver** (`unverified_timestamps`) — timestamps without inter-annotator agreement
- **Bronze** (`no_timestamps`) — clip-level tags only; an empty timestamp list here is *correct*,
  not a parsing bug


In [ ]:
import torch, torchaudio, soundfile as sf, librosa
import torch.nn as nn, torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

# Seed torch too. random/np were seeded in Section 0, but model init and the DataLoader
# sampler draw from torch's generator, which otherwise starts from OS entropy - so two runs
# of the same config would not be comparable. DataLoader workers inherit from this seed.
torch.manual_seed(CFG["seed"]); torch.cuda.manual_seed_all(CFG["seed"])
print("seeded torch:", CFG["seed"])

def decode_audio(a, target_sr=CFG["sr"]):
    w = sr = None
    if isinstance(a, dict):
        if a.get("array") is not None:
            w, sr = np.asarray(a["array"], dtype=np.float32), a["sampling_rate"]
        elif a.get("bytes"):
            w, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32", always_2d=False)
        elif a.get("path"):
            w, sr = sf.read(a["path"], dtype="float32", always_2d=False)
    elif hasattr(a, "get_all_samples"):
        s = a.get_all_samples(); w, sr = s.data.numpy().astype(np.float32), int(s.sample_rate)
    elif hasattr(a, "path"):
        w, sr = sf.read(a.path, dtype="float32", always_2d=False)
    if w is None:
        raise ValueError(f"cannot decode {type(a)}")
    w = np.asarray(w, dtype=np.float32)
    if w.ndim > 1:
        w = w.mean(axis=0) if w.shape[0] < w.shape[1] else w.mean(axis=1)
    if sr != target_sr:
        w = librosa.resample(w, orig_sr=sr, target_sr=target_sr)
    return np.ascontiguousarray(w, dtype=np.float32)

def _f(x):
    try: return float(x)
    except: return None

def spans_from(ex):
    out = []
    for s in (ex.get("NoiseSubCategoryTimeStamp") or []):
        st, en = _f(s.get("start")), _f(s.get("end"))
        if st is not None and en is not None and en > st:
            out.append((st, en, s.get("category")))
    return sorted(out)

def build_labels(spans, n_frames, has_strong):
    lab = np.zeros((1 + CFG["n_cat"], n_frames), dtype=np.uint8)
    if has_strong:
        for st, en, cat in spans:
            a, b = max(0, int(round(st * 100))), min(n_frames, int(round(en * 100)))
            if b <= a: continue
            lab[0, a:b] = 1
            ci = CAT2IDX.get(cat)
            if ci is not None: lab[1 + ci, a:b] = 1
    return lab

def clip_tags(ex):
    y = np.zeros(1 + CFG["n_cat"], dtype=np.float32)
    cats = ex.get("NoiseCategory") or []
    if len(cats): y[0] = 1.0
    for c in cats:
        ci = CAT2IDX.get(c)
        if ci is not None: y[1 + ci] = 1.0
    return y


### 2b. Decode once and rasterise labels

This is the single pass over the streamed audio. For every kept clip it:

1. **Decodes** the raw bytes to a 16 kHz mono waveform, stored as **int16** in memory.
2. **Rasterises labels** to a 100 Hz `(8, T)` array — channel 0 = "any noise", channels 1–7 =
   the seven categories — plus the clip-level `tags`.

Everything lands in a single `rows` list that Section 3 splits into train/val without re-reading
audio.


In [ ]:
counts_done = {k: 0 for k in MAX_PER_QUALITY}
rows = []; t0 = time.time()

def done():
    for q, cap in MAX_PER_QUALITY.items():
        if cap is None: return False
        if cap and counts_done[q] < cap: return False
    return True

for i, ex in enumerate(tqdm(raw, desc="loading")):
    q = ex["annotationQuality"]; cap = MAX_PER_QUALITY.get(q, 0)
    if cap == 0: continue
    if cap is not None and counts_done[q] >= cap:
        if done(): break
        continue
    try:
        wav = decode_audio(ex["audio"])
    except Exception as e:
        if i < 5: print("decode failed row", i, e)
        continue
    if wav is None or len(wav) < CFG["sr"] * 0.2: continue
    spans = spans_from(ex)
    has_strong = q in ("verified_timestamps", "unverified_timestamps")

    nf = 1 + len(wav) // CFG["hop"]
    rows.append(dict(
        wav=(np.clip(wav, -1, 1) * 32767).astype(np.int16),
        lab=build_labels(spans, nf, has_strong),
        tags=clip_tags(ex),
        quality=q, has_strong=bool(has_strong),
    ))
    counts_done[q] += 1

print(counts_done, f"| {len(rows)} rows | {(time.time()-t0)/60:.1f} min")
if counts_done.get("verified_timestamps", 0) == 0:
    print("!! ZERO gold - detection will be weak and the Track 2 banks thin")


---
## 3. Dataset

Fixed 5-second windows. Five rather than ten halves the WavLM sequence length and lets the batch
double — the largest single speed lever in this notebook.

One subtle correctness point, handled in `__getitem__`: a random crop can cut the event out of
the window while the stored clip-level tag still says it is present. That contradiction poisons
the weak loss, so for clips that have timestamps we **recompute the tag from the cropped
labels**. Bronze has no timestamps, so its stored tag is all we have.

In [ ]:
from torch.utils.data import Dataset, DataLoader

# ---- bucket the in-memory rows by annotation tier ----
by_q = defaultdict(list)
for r in rows:
    by_q[r["quality"]].append(r)
gold   = by_q["verified_timestamps"]      # trustworthy onset/offset
silver = by_q["unverified_timestamps"]    # timestamps without agreement (down-weighted)
bronze = by_q["no_timestamps"]            # clip-level tags only (weak / attention head)
print(f"gold {len(gold)} | silver {len(silver)} | bronze {len(bronze)}")

# per-sample loss weight by tier
QW = {"verified_timestamps": 1.0,
      "unverified_timestamps": CFG["silver_weight"],
      "no_timestamps": 0.2}

def prep_wav(w):
    w = np.asarray(w, dtype=np.float32)
    return (w - w.mean()) / (w.std() + 1e-5)

def _labels_20ms(lab, s):
    # crop the 100 Hz label window then max-pool 10 ms -> 20 ms to match FRAMES_OUT
    lab = lab[:, s // CFG["hop"]: s // CFG["hop"] + FRAMES_100HZ]
    if lab.shape[1] < FRAMES_100HZ:
        lab = np.pad(lab, ((0, 0), (0, FRAMES_100HZ - lab.shape[1])))
    lab = lab[:, :FRAMES_100HZ]
    return lab.reshape(lab.shape[0], FRAMES_OUT, CFG["time_pool"]).max(axis=2)

class SEDData(Dataset):
    def __init__(self, rows, train=True):
        self.rows, self.train = rows, train
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        m = self.rows[i]
        wav = m["wav"].astype(np.float32) / 32767.0
        lab = m["lab"].astype(np.float32)          # (8, T100)
        tags = m["tags"].astype(np.float32)        # (8,)

        # crop/pad the waveform to a fixed 5 s window
        if len(wav) > N_SAMP:
            s = random.randint(0, len(wav) - N_SAMP) if self.train else 0
            wav = wav[s:s + N_SAMP]; valid = N_SAMP
        else:
            s = 0; valid = len(wav); wav = np.pad(wav, (0, N_SAMP - len(wav)))

        lab20 = _labels_20ms(lab, s)               # (8, FRAMES_OUT)
        strong = 1.0 if m["has_strong"] else 0.0
        # a crop can drop the event while the stored tag still says present -> recompute
        # the clip tag from the cropped labels for clips that HAVE timestamps.
        if m["has_strong"]:
            tags = (lab20.max(axis=1) > 0).astype(np.float32)

        mask = np.zeros(FRAMES_OUT, dtype=np.float32)
        n_valid = min(FRAMES_OUT, max(1, int(math.ceil(valid / CFG["sr"] / FRAME_SEC))))
        mask[:n_valid] = 1.0

        return dict(
            wav=torch.from_numpy(prep_wav(wav)),
            lab=torch.from_numpy(lab20.astype(np.float32)),
            tags=torch.from_numpy(tags),
            mask=torch.from_numpy(mask),
            strong=torch.tensor(strong, dtype=torch.float32),
            w=torch.tensor(float(QW.get(m["quality"], 1.0)), dtype=torch.float32),
        )

# ---- split: gold gives the (strong) validation set; everything trains ----
random.Random(CFG["seed"]).shuffle(gold)
n_val = max(1, int(len(gold) * CFG["val_frac"]))
val_rows, train_rows = gold[:n_val], gold[n_val:] + silver + bronze
random.Random(CFG["seed"]).shuffle(train_rows)
if TRAIN_LIMIT is not None:
    train_rows = train_rows[:TRAIN_LIMIT]
print(f"train {len(train_rows)} | val {len(val_rows)} (gold only)")

train_dl = DataLoader(SEDData(train_rows, True), batch_size=T1_BS, shuffle=True,
                      num_workers=CFG["num_workers"], drop_last=True, pin_memory=True)
val_dl = DataLoader(SEDData(val_rows, False), batch_size=T1_BS, shuffle=False,
                    num_workers=CFG["num_workers"], pin_memory=True)

b = next(iter(train_dl))
print("batch:", {k: tuple(v.shape) for k, v in b.items()})


---
## 4. Model

WavLM-base+ emits 50 Hz frames — exactly the 20 ms label grid — so no interpolation is needed
during training. The last two transformer layers are dropped: Schmid et al. (ICASSP 2025) found
final layers overfit the pretraining objective while earlier ones transfer better.

Two heads sit on the BiGRU. The **frame head** gives per-frame event probabilities. The
**attention-pooled clip head** gives a per-clip probability, and it is the only way Bronze
clips can contribute a gradient at all.

In [ ]:
from transformers import AutoModel

class WavLMSED(nn.Module):
    input_key = "wav"
    def __init__(self, name=ENCODER_NAME, n_out=8, rnn_dim=256, drop_layers=DROP_LAST_LAYERS):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        if drop_layers > 0:
            self.enc.encoder.layers = self.enc.encoder.layers[:-drop_layers]
        d = self.enc.config.hidden_size
        self.rnn = nn.GRU(d, rnn_dim, 2, batch_first=True, bidirectional=True, dropout=0.1)
        self.strong = nn.Linear(2 * rnn_dim, n_out)
        self.att = nn.Linear(2 * rnn_dim, n_out)
    def forward(self, x, mask=None):
        h = self.enc(x).last_hidden_state
        T = mask.shape[1] if mask is not None else h.shape[1]
        if h.shape[1] != T:                       # conv stride rounding, off by a frame or two
            h = F.interpolate(h.transpose(1, 2), size=T, mode="linear",
                              align_corners=False).transpose(1, 2)
        h, _ = self.rnn(h)
        frame = torch.sigmoid(self.strong(h))
        a = self.att(h)
        if mask is not None:
            a = a.masked_fill(mask.unsqueeze(-1) < 0.5, -1e4)
        a = torch.softmax(a, dim=1)
        clip = (frame * a).sum(dim=1).clamp(1e-6, 1 - 1e-6)
        return frame.transpose(1, 2), clip

_m = WavLMSED().to(device)
print(f"{sum(p.numel() for p in _m.parameters())/1e6:.1f}M params")
with torch.no_grad():
    _f_, _c_ = _m(b["wav"].to(device), b["mask"].to(device))
print("frame", tuple(_f_.shape), "clip", tuple(_c_.shape))
del _m; torch.cuda.empty_cache()

---
## 5. Metrics — official scorer, verbatim

Copied from the Evaluation page. Do not "improve" these; the point is that local == leaderboard.

Three things that drive tuning:
- tolerance is `max(0.20 × d, 0.05)` — the floor is **50 ms**
- matching is **global closest-first**, not first-fit
- Dice is macro per clip, and **a clip with no reference and no prediction scores 1.0** — so a
  single false alarm on a clean clip costs that clip's entire Dice point

**Combined = Event F1 + Segment Dice**, max 2.0.

In [ ]:
from scipy.ndimage import median_filter

def match_events(ref_events, pred_events, tolerance_frac=0.20):
    matched_ref, matched_pred, candidates = set(), set(), []
    for ri, (r_on, r_off) in enumerate(ref_events):
        tol = max(tolerance_frac * (r_off - r_on), 0.05)
        for pi, (p_on, p_off) in enumerate(pred_events):
            if abs(p_on - r_on) <= tol and abs(p_off - r_off) <= tol:
                candidates.append((abs(p_on - r_on) + abs(p_off - r_off), ri, pi))
    for _, ri, pi in sorted(candidates):
        if ri not in matched_ref and pi not in matched_pred:
            matched_ref.add(ri); matched_pred.add(pi)
    tp = len(matched_ref)
    return tp, len(pred_events) - tp, len(ref_events) - tp

def event_based_f1(ref_data, pred_data):
    TP = FP = FN = 0
    for cid, ref_events in ref_data.items():
        tp, fp, fn = match_events(ref_events, pred_data.get(cid, []))
        TP += tp; FP += fp; FN += fn
    for cid in pred_data:
        if cid not in ref_data: FP += len(pred_data[cid])
    p = TP / (TP + FP) if (TP + FP) else 0.0
    r = TP / (TP + FN) if (TP + FN) else 0.0
    return ((2 * p * r / (p + r)) if (p + r) else 0.0), p, r, TP, FP, FN

def events_to_frames(events, max_time, frame_len=0.01):
    n = int(max_time / frame_len) + 1
    mask = [0] * n
    for on, off in events:
        for i in range(int(on / frame_len), min(int(off / frame_len) + 1, n)):
            mask[i] = 1
    return mask

def segment_dice(ref_data, pred_data):
    scores = []
    for cid, ref_events in ref_data.items():
        pred_events = pred_data.get(cid, [])
        all_ev = ref_events + pred_events
        if not all_ev:
            scores.append(1.0); continue
        max_time = max(off for _, off in all_ev) + 0.5
        rm, pm = events_to_frames(ref_events, max_time), events_to_frames(pred_events, max_time)
        inter = sum(r & p for r, p in zip(rm, pm)); total = sum(rm) + sum(pm)
        scores.append(1.0 if total == 0 else 2.0 * inter / total)
    return sum(scores) / len(scores) if scores else 0.0

def prob_to_events(p, thr=0.5, med=7, frame_sec=FRAME_SEC, min_dur=0.05, merge_gap=0.10):
    b_ = (np.asarray(p) >= thr).astype(np.uint8)
    if med > 1: b_ = median_filter(b_, size=med, mode="nearest")
    ev, start = [], None
    for i, v in enumerate(b_):
        if v and start is None: start = i
        elif not v and start is not None:
            ev.append([start * frame_sec, i * frame_sec]); start = None
    if start is not None: ev.append([start * frame_sec, len(b_) * frame_sec])
    out = []
    for e in ev:
        if out and e[0] - out[-1][1] <= merge_gap: out[-1][1] = e[1]
        else: out.append(e)
    return [e for e in out if e[1] - e[0] >= min_dur]

@torch.no_grad()
def cache_posteriors(model, dl):
    # posteriors do not change when sweeping post-processing, so run the model ONCE
    model.eval(); cache = []
    for batch in dl:
        frame, clip = model(batch["wav"].to(device), batch["mask"].to(device))
        for p, cp, g, msk in zip(frame[:, 0].float().cpu().numpy(),
                                 clip[:, 0].float().cpu().numpy(),
                                 batch["lab"][:, 0].numpy(), batch["mask"].numpy()):
            nv = int(msk.sum()); cache.append((p[:nv].copy(), g[:nv].copy(), float(cp)))
    ref = {f"c{i}": [tuple(e) for e in prob_to_events(g, 0.5, 1, min_dur=0.0, merge_gap=0.0)]
           for i, (_, g, _) in enumerate(cache)}
    return cache, ref

def score_cached(cache, ref, thr=0.5, med=7, min_dur=0.05, merge_gap=0.10, clip_gate=0.0):
    # clip_gate: if the attention-pooled clip probability is below this, emit NO events.
    # Dice is macro per clip and a clip with no reference and no prediction scores 1.0, so
    # staying silent on a clip you believe is clean buys a whole Dice point. Suppressing a
    # real event costs only its share of F1. The asymmetry is large; sweep the gate.
    pred = {}
    for i, (p, _, cp) in enumerate(cache):
        pred[f"c{i}"] = ([] if cp < clip_gate else
                         [tuple(e) for e in prob_to_events(p, thr, med, min_dur=min_dur,
                                                           merge_gap=merge_gap)])
    f1, prec, rec, TP, FP, FN = event_based_f1(ref, pred)
    dice = segment_dice(ref, pred)
    return dict(f1=f1, dice=dice, score=f1 + dice, precision=prec, recall=rec,
                tp=TP, fp=FP, fn=FN)

def evaluate(model, dl, **kw):
    cache, ref = cache_posteriors(model, dl)
    return score_cached(cache, ref, **kw)

---
## 6. Two-stage fine-tuning

**Stage 1** freezes the encoder and trains only the GRU and heads. **Stage 2** unfreezes with a
much smaller learning rate on the encoder (5e-5) than on the heads (3e-4).

Skipping stage 1 is the classic failure: a large pretrained encoder fine-tuned end-to-end from
step 0 overfits immediately and scores below a from-scratch CRNN. That failure is the entire
reason the ATST-SED paper exists.

### What label each tier actually carries

The decode pass (Section 2) writes two supervision arrays per clip:

- `lab` — a **frame-level** `(8, T)` mask (channel 0 = "any noise", 1–7 = categories), rasterised
  from the timestamps. Only clips with timestamps get a non-zero `lab`; for **Bronze** it is all
  zeros because there are no timestamps to rasterise.
- `tags` — a **clip-level** `(8,)` vector: "does this clip contain noise / which categories",
  with no timing. **Every** tier has this, including Bronze.

So the three tiers differ only in *what* label they own:

| tier | `quality` | frame label `lab` | clip tag `tags` | `strong` flag | loss weight `w` |
|---|---|---|---|---|---|
| **Gold** | `verified_timestamps` | ✅ trustworthy | ✅ | 1 | 1.0 |
| **Silver** | `unverified_timestamps` | ✅ no agreement | ✅ | 1 | 0.4 (`silver_weight`) |
| **Bronze** | `no_timestamps` | ❌ all zeros | ✅ | 0 | 0.2 (unused, see below) |

The dataset (Section 3) attaches the tier to each sample as two tensors, `strong` (1 for
Gold/Silver, 0 for Bronze) and `w` (the per-tier weight). The loss reads those two numbers to
decide how each clip participates.

### How the three loss terms consume the labels

Each step computes `loss = l_strong + l_weak + cw * l_cons`:

**1. Frame loss `l_strong` — timing supervision (Gold + Silver only).**
Per-frame BCE between the frame head and `lab`, then reduced with `* strong * w`:
```python
l_strong = (ls * strong * w).sum() / (strong * w).sum().clamp(min=1e-6)
```
The `* strong` factor **zeros Bronze out entirely** (it has no frame label). Among the survivors
`* w` makes **Gold count 1.0 and Silver 0.4**, and the denominator normalises to a weighted
average over only the timestamped clips. This is the *only* term that learns onset/offset timing.

**2. Weak loss `l_weak` — clip-tag supervision (all tiers, this is Bronze's main path).**
```python
l_weak = bce_pos(c_s, tags).mean()      # no `strong` gate -> Bronze included
```
`c_s` is the **attention-pooled** clip probability, `clip = (frame * a).sum(dim=1)`, where the
model *learns where* the evidence is via softmax attention `a`. Comparing that pooled value to the
Bronze tag is Multiple-Instance Learning: the bag label ("dog present somewhere") pushes the frame
probabilities up at the time steps the attention already favours, without ever being told the
exact interval. Because there is **no `* strong` mask** here, Bronze flows through fully — this is
how a clip with no timestamps still trains the frame head (indirectly, through the pooling).

**3. Consistency loss `l_cons` — mean-teacher (all tiers, tier-agnostic).**
```python
l_cons = ((f_s - f_t)**2 * m3).sum()/... + ((c_s - c_t)**2).mean()
```
Every clip — Gold, Silver, Bronze — is augmented (`wav_augment`) for the student and passed clean
to the EMA teacher, and the two must agree on the frame and clip outputs. Bronze therefore also
regularises the frame head by demanding stable predictions on real audio the Gold set never
covered.

Net effect per tier: **Gold** drives precise timing at full strength, **Silver** does the same at
40%, and **Bronze** contributes only through the clip-tag (MIL) head and the mean-teacher
consistency signal — never directly to frame-level timing. (Bronze's `w = 0.2` is effectively
unused, since the only place `w` is applied, `l_strong`, already excludes Bronze via `strong = 0`.)

### Other mechanics active in the loop

- **Mean teacher** with a **ramped EMA decay**. At a fixed 0.999 the teacher sits near random
  init for the first ~1000 steps, exactly while the consistency weight ramps up.
- **Positive-weighted BCE** — noise frames are a small minority, so plain BCE under-fires.
- **Mixup** with **union** labels: a mixed clip genuinely contains both event sets, so the
  labels are `max`, not a soft blend.
- Losses computed in **fp32 outside the autocast region**. `BCELoss` is explicitly unsafe to
  autocast, and an fp16 sigmoid can saturate to exactly 1.0.


In [ ]:
import copy
from torch.amp import autocast, GradScaler

def wav_augment(x, n=2, tmax=None):
    tmax = tmax or int(0.15 * CFG["sr"])
    x = x.clone()
    for b_ in range(x.shape[0]):
        for _ in range(n):
            t = random.randint(0, tmax); t0 = random.randint(0, max(0, x.shape[1] - t))
            x[b_, t0:t0 + t] = 0
    return x

def mixup(inp, lab, tags, alpha=CFG["mixup_alpha"]):
    lam = float(np.random.beta(alpha, alpha)); lam = max(lam, 1 - lam)
    perm = torch.randperm(inp.size(0), device=inp.device)
    return (lam * inp + (1 - lam) * inp[perm],
            torch.maximum(lab, lab[perm]), torch.maximum(tags, tags[perm]))

def bce_pos(p, target, pw=None):
    pw = CFG["pos_weight"] if pw is None else pw
    p = p.clamp(1e-6, 1 - 1e-6)
    return -(pw * target * torch.log(p) + (1 - target) * torch.log(1 - p))

def rampup(ep, n):
    return 1.0 if n == 0 else float(np.exp(-5 * (1 - np.clip(ep / n, 0, 1)) ** 2))

@torch.no_grad()
def ema_update(stu, tea, decay, step=None):
    if step is not None:
        decay = min(1 - 1 / (step + 1), decay)      # else the teacher stays at random init
    for ts, ss in zip(tea.state_dict().values(), stu.state_dict().values()):
        if ts.dtype.is_floating_point: ts.mul_(decay).add_(ss.detach(), alpha=1 - decay)
        else: ts.copy_(ss)

state = dict(best=-1.0, gstep=0)

def run_epochs(stu, tea, opt, sched, epochs, scaler, tag=""):
    for ep in range(epochs):
        stu.train(); tea.train()
        cw = CFG["max_cons_w"] * rampup(ep, CFG["rampup_epochs"])
        acc = defaultdict(float); t0 = time.time()
        for batch in train_dl:
            inp = batch["wav"].to(device, non_blocking=True)
            lab = batch["lab"].to(device); tags = batch["tags"].to(device)
            mask = batch["mask"].to(device)
            strong = batch["strong"].to(device); w = batch["w"].to(device)
            if random.random() < CFG["mixup_prob"]:
                inp, lab, tags = mixup(inp, lab, tags)
            with autocast("cuda"):
                f_s, c_s = stu(wav_augment(inp), mask)
                with torch.no_grad():
                    f_t, c_t = tea(inp, mask)
            m3 = mask.unsqueeze(1)
            f_s = f_s.float().clamp(1e-6, 1 - 1e-6); c_s = c_s.float().clamp(1e-6, 1 - 1e-6)
            ls = (bce_pos(f_s, lab) * m3).sum(dim=(1, 2)) / (m3.sum(dim=(1, 2)) * lab.shape[1] + 1e-6)
            l_strong = (ls * strong * w).sum() / (strong * w).sum().clamp(min=1e-6)
            l_weak = bce_pos(c_s, tags).mean()
            f_t, c_t = f_t.float(), c_t.float()
            l_cons = (((f_s - f_t) ** 2) * m3).sum() / (m3.sum() * lab.shape[1] + 1e-6) \
                     + ((c_s - c_t) ** 2).mean()
            loss = l_strong + l_weak + cw * l_cons
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(stu.parameters(), 5.0)
            scaler.step(opt); scaler.update()
            if sched is not None: sched.step()
            state["gstep"] += 1
            ema_update(stu, tea, CFG["ema_decay"], step=state["gstep"])
            acc["s"] += float(l_strong.detach()); acc["w"] += float(l_weak.detach())
            acc["c"] += float(l_cons.detach()); acc["n"] += 1
        n = max(1, acc["n"])
        ms, mt = evaluate(stu, val_dl), evaluate(tea, val_dl)
        print(f"{tag}ep{ep:02d} [{(time.time()-t0)/60:.1f}m] s {acc['s']/n:.3f} w {acc['w']/n:.3f} "
              f"c {acc['c']/n:.3f} | stu {ms['score']:.4f} (F1 {ms['f1']:.3f} D {ms['dice']:.3f}) "
              f"| tea {mt['score']:.4f}")
        if max(ms["score"], mt["score"]) > state["best"]:
            state["best"] = max(ms["score"], mt["score"])
            which = "student" if ms["score"] >= mt["score"] else "teacher"
            torch.save({"model": (stu if which == "student" else tea).state_dict(),
                        "which": which, "score": state["best"], "cfg": CFG}, CKPT)
            print(f"   saved {which} {state['best']:.4f}")

student = WavLMSED().to(device)
teacher = copy.deepcopy(student).to(device)
for p in teacher.parameters(): p.requires_grad_(False)
scaler = GradScaler("cuda")
head = [p for n_, p in student.named_parameters() if not n_.startswith("enc.")]

print(f"=== stage 1: encoder frozen, {STAGE1_EPOCHS} epochs ===")
for p in student.enc.parameters(): p.requires_grad_(False)
run_epochs(student, teacher,
           torch.optim.AdamW(head, lr=LR_HEAD, weight_decay=CFG["weight_decay"]),
           None, STAGE1_EPOCHS, scaler, tag="s1 ")

print(f"=== stage 2: unfrozen, enc lr {LR_ENC}, {STAGE2_EPOCHS} epochs ===")
for p in student.enc.parameters(): p.requires_grad_(True)
opt2 = torch.optim.AdamW([{"params": student.enc.parameters(), "lr": LR_ENC},
                          {"params": head, "lr": LR_HEAD * 0.3}],
                         weight_decay=CFG["weight_decay"])
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=STAGE2_EPOCHS * max(1, len(train_dl)))
run_epochs(student, teacher, opt2, sched2, STAGE2_EPOCHS, scaler, tag="s2 ")
print("\nbest Combined:", state["best"], "/ 2.0")
del student, teacher; torch.cuda.empty_cache()

---
## 7. Post-processing sweep

The model runs **once** over the validation split and every configuration is scored on the
cached posteriors. Re-running the forward pass for each of ~500 configurations would waste the
better part of an hour for identical results.

Four knobs: `threshold`, `median filter length`, `min_dur`, and the **clip gate**.

The clip gate is the one worth explaining. Dice is macro-averaged per clip, and a clip with no
reference *and* no prediction scores 1.0. Staying silent on a clip the model believes is clean
buys a whole Dice point; suppressing a real event costs only that event's share of F1. The
asymmetry is large, so the gate is swept alongside everything else.

In [ ]:
ck = torch.load(CKPT, map_location=device, weights_only=False)
t1 = WavLMSED().to(device); t1.load_state_dict(ck["model"]); t1.eval()
print("loaded", ck["which"], f"{ck['score']:.4f}")

t0 = time.time(); cache, ref = cache_posteriors(t1, val_dl)
print(f"cached {len(cache)} clips in {time.time()-t0:.0f}s")

n_empty_ref = sum(1 for v in ref.values() if not v)
print(f"val clips with NO reference events: {n_empty_ref}/{len(ref)} "
      f"({100*n_empty_ref/max(1,len(ref)):.1f}%) - each is a free Dice point if we stay silent")

results = []
for thr in [0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7]:
    for med in [1, 3, 5, 9, 15, 21]:
        for min_dur in [0.05, 0.12, 0.25]:
            for gate in [0.0, 0.3, 0.5, 0.7]:
                m = score_cached(cache, ref, thr=thr, med=med, min_dur=min_dur, clip_gate=gate)
                results.append((m["score"], m["f1"], m["dice"], thr, med, min_dur, gate,
                                m["precision"], m["recall"]))
results.sort(reverse=True)
print(f"{'Comb':>7}{'F1':>7}{'Dice':>7}{'thr':>6}{'med':>5}{'mind':>7}{'gate':>6}{'P':>7}{'R':>7}")
for r in results[:12]:
    print(f"{r[0]:7.4f}{r[1]:7.3f}{r[2]:7.3f}{r[3]:6.2f}{r[4]:5d}{r[5]:7.2f}{r[6]:6.2f}"
          f"{r[7]:7.3f}{r[8]:7.3f}")

T1_THR, T1_MED, T1_MINDUR, T1_GATE = (results[0][3], results[0][4], results[0][5], results[0][6])
ck.update(thr=float(T1_THR), med=int(T1_MED), min_dur=float(T1_MINDUR), gate=float(T1_GATE))
torch.save(ck, CKPT)
print(f"\nthr={T1_THR} med={T1_MED} min_dur={T1_MINDUR} gate={T1_GATE} "
      f"-> Combined {results[0][0]:.4f} / 2.0")
no_gate = max(r[0] for r in results if r[6] == 0.0)
print(f"gate contributes +{results[0][0]-no_gate:.4f} over the best ungated config")
if results[0][6] < results[0][7] - 0.1:
    print("!! precision << recall: you are firing on quiet clips, which costs a full Dice "
          "point per clean clip. Prefer a higher thr even at some F1 cost.")

---
## 8. Inference helpers

`t1_posteriors` runs a whole clip through the model in one pass. The CNN front-end is
convolutional and the GRU takes any length, so no windowing is needed below two minutes — which
also avoids a train/inference mismatch, since normalisation is computed over the whole clip in
both places.

`export_track1` turns posteriors into events: threshold, median filter, minimum duration, gap
merging, then the clip gate — returning the onset/offset list used for the submission.


In [ ]:
MAX_ONESHOT_SEC = 120.0

MIN_SAMPLES = 640          # WavLM conv needs >= 400 samples; keep a margin

@torch.no_grad()
def t1_posteriors(wav, sr=CFG["sr"], model=None):
    # whole-clip forward; normalisation matches training exactly.
    # returns (post (8, T20), clip_prob_any_noise)
    model = t1 if model is None else model
    wav = np.asarray(wav, dtype=np.float32)
    if len(wav) < MIN_SAMPLES:
        wav = np.pad(wav, (0, MIN_SAMPLES - len(wav)))
    dur = len(wav) / sr
    n_out = max(1, int(math.ceil(dur / FRAME_SEC)))
    if dur <= MAX_ONESHOT_SEC:
        x = torch.from_numpy(prep_wav(wav)).unsqueeze(0).to(device)
        fr, cl = model(x, None)
        p = fr[0].float().cpu().numpy(); cp = float(cl[0, 0])
        if p.shape[1] != n_out:
            p = np.stack([np.interp(np.linspace(0, 1, n_out),
                                    np.linspace(0, 1, p.shape[1]), r) for r in p])
        return p, cp
    win, hp = int(30 * sr), int(28 * sr)
    starts = list(range(0, max(1, len(wav) - win + hp), hp)) or [0]
    acc = np.zeros((8, n_out)); cnt = np.zeros(n_out) + 1e-9; cps = []
    for s0 in starts:
        seg = wav[s0:s0 + win]
        if len(seg) < MIN_SAMPLES: continue
        x = torch.from_numpy(prep_wav(seg)).unsqueeze(0).to(device)
        fr, cl = model(x, None)
        p = fr[0].float().cpu().numpy(); cps.append(float(cl[0, 0]))
        off = int(round(s0 / sr / FRAME_SEC))
        for j in range(p.shape[1]):
            k = off + j
            if 0 <= k < n_out: acc[:, k] += p[:, j]; cnt[k] += 1
    return acc / cnt, (max(cps) if cps else 0.0)

def export_track1(clip_id, wav, sr=CFG["sr"], thr=None, med=None, min_dur=None, gate=None):
    thr = T1_THR if thr is None else thr
    med = T1_MED if med is None else med
    min_dur = T1_MINDUR if min_dur is None else min_dur
    gate = T1_GATE if gate is None else gate
    post, clip_p = t1_posteriors(wav, sr)
    dur = max(len(wav) / sr, MIN_SAMPLES / sr)
    spans = []
    # Clip-level gate: believe the clip is clean -> emit nothing -> full Dice point.
    events_iter = [] if clip_p < gate else prob_to_events(post[0], thr, med, min_dur=min_dur)
    for on, off in events_iter:
        on, off = float(max(0.0, on)), float(min(dur, off))
        if off - on < 0.05: continue
        spans.append([on, off])
    return {"clip_id": clip_id,
            "events": [{"onset": round(float(a), 3), "offset": round(float(b), 3)}
                       for a, b in spans]}

for nm, _w in [("3s", np.random.randn(CFG["sr"] * 3).astype(np.float32) * 0.05),
               ("0.3s", np.random.randn(int(CFG["sr"] * 0.3)).astype(np.float32) * 0.05),
               ("silence", np.zeros(CFG["sr"] * 2, dtype=np.float32))]:
    r = export_track1(f"__smoke_{nm}__", _w)
    print(f"{nm:>8}: {len(r['events'])} events")


---
## 9. Track 1 submission

Official format: ZIP with `predictions.jsonl` **at the root**, key `clip_id`, events carrying only
onset/offset (Track 1 is class-agnostic). Every eval clip appears exactly once; clips with nothing
detected get `[]`, not a missing line. Extra clip_ids not in the reference count all their events
as false positives. Limits: 5 submissions/day, 100 total.


In [ ]:
AUD = (".wav", ".flac", ".mp3", ".ogg")

def read_audio(p):
    w, sr = sf.read(str(p), dtype="float32", always_2d=False)
    if w.ndim > 1: w = w.mean(axis=1)
    if sr != CFG["sr"]: w = librosa.resample(w, orig_sr=sr, target_sr=CFG["sr"])
    return w

T1_TEST_DIR = Path("/kaggle/input/indoml-track1-test")   # input_data from the Files tab
T1_JSONL, T1_ZIP = WORK / "predictions.jsonl", WORK / "submission_track1.zip"

if T1_TEST_DIR.exists():
    files = sorted([p for p in T1_TEST_DIR.rglob("*") if p.suffix.lower() in AUD])
    print(len(files), "test clips")
    with open(T1_JSONL, "w", encoding="utf-8") as f:
        for p in tqdm(files):
            rec = export_track1(p.stem, read_audio(p))
            f.write(json.dumps({"clip_id": p.stem, "events": rec["events"]},
                               ensure_ascii=False) + "\n")
    with zipfile.ZipFile(T1_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(T1_JSONL, "predictions.jsonl")          # at ZIP ROOT, no enclosing folder
    n = sum(1 for _ in open(T1_JSONL))
    empty = sum(1 for l in open(T1_JSONL) if not json.loads(l)["events"])
    print(f"{n} lines ({empty} with no events) -> {T1_ZIP}")
    print(open(T1_JSONL).readline().strip()[:140])
    print("!! verify clip_id matches the eval metadata; we use the filename stem")
else:
    print(f"{T1_TEST_DIR} not found. Accept terms under 'My Submissions' on Codabench, "
          f"download input_data from the Files tab, attach as a Kaggle Dataset, re-run.")
    print("(read_audio is defined above, so the Track 2 cells below still work.)")

---
## Appendix — the model & training, explained simply

New to this domain? Read this first. It skips the jargon and uses everyday analogies.

### The big goal

You have a speech clip. Somewhere in it a **dog barks**, a **horn honks**, a **baby cries**. Your
job: point at *exactly when* each noise happens — "the horn is from 1.2s to 3.8s." Everything below
is machinery to do that well.

---

## Part 1 — The Model (the "brain" that listens)

Think of the model as an assembly line with 4 stations. Audio goes in one end, "noise happening
now? yes/no" comes out the other.

### Station 1 — WavLM (the ears) 👂

Raw audio is just a list of numbers (16,000 per second). Meaningless to look at directly.

**WavLM** is a pre-trained "ear." It already listened to *thousands of hours* of audio before you
ever touched it, so it learned what dogs, horns, speech, etc. *sound like*. You downloaded it
already knowing.

- **Input:** raw audio.
- **Output:** every 20 milliseconds (50 times per second) it produces a "summary" of what that tiny
  slice sounds like — a list of 768 numbers describing it.

> Analogy: instead of raw pixels, WavLM hands you "this frame contains fur, four legs, a wagging
> tail" — meaningful descriptions.

### Station 2 — BiGRU (the memory) 🧠

WavLM describes each 20 ms slice *on its own*. But a bark isn't one slice — it *builds up and
fades*. To know where a sound starts and stops, you need to look at neighbouring slices too.

The **BiGRU** reads the sequence of slices **both forwards and backwards**, so each moment
"remembers" what came just before and just after. This is what sharpens the *start* and *end* of
each event.

> Analogy: reading a sentence. You understand each word better because you saw the words around it.

### Stations 3 & 4 — Two heads (two answers) 🎯

From the BiGRU's output, we ask **two different questions**:

**Head A — the Frame head → "Is there noise RIGHT NOW?"**
For every 20 ms slice it outputs a probability (0 to 1). String these together and you get a
timeline:
```
time:  0.0  0.5  1.0  1.5  2.0  2.5
noise:  no   no   YES  YES  YES  no
```
→ "there's a noise from 1.0s to 2.0s." **This is what actually produces your answer.**

**Head B — the Clip head → "Is there noise ANYWHERE in this clip?"**
Just one yes/no for the whole clip, ignoring *when*. Seems useless… but it's the key to Part 2.

---

## Part 2 — Training (teaching the brain)

Training = showing the model thousands of examples and correcting its mistakes until it gets good.
Here's the twist that confuses most people:

### Your labelled data comes in 3 quality levels

Like three kinds of homework, marked by how carefully they were checked:

| Tier | What you're told | Analogy |
|---|---|---|
| 🥇 **Gold** | "Dog barks from 1.2s to 3.8s" — *exact times, double-checked* | Answer key verified by the teacher |
| 🥈 **Silver** | "Dog barks from ~1.2s to ~3.8s" — *times given, not double-checked* | A classmate's answers — probably right |
| 🥉 **Bronze** | "There's a dog somewhere in this clip" — *no times at all* | Just told the topic, not the answer |

The problem: **Gold is rare** (expensive to make), Bronze is plentiful. You want to use *all* of it,
but you can't trust it equally.

### How each tier teaches the model

This is where the **two heads** pay off:

**🥇 Gold & 🥈 Silver → teach the Frame head (the "when").**
Because they have exact times, you can tell the model: "at 1.2s you should have said YES, but you
said no — wrong, adjust." Gold corrections count **full strength**; Silver counts **40%** (since we
trust it less). This is the only part that learns precise *timing*.

**🥉 Bronze → teaches the Clip head (the "whether").**
Bronze has no times, so it *can't* correct the frame timeline. But it *can* answer "is there a dog
anywhere? yes." So we compare Bronze against **Head B** instead.

The clever bit — **Head B is not independent; it is literally computed *from* Head A's timeline.**
Head A gives a probability at every 20 ms slice, and Head B collapses that whole timeline into one
number by taking a **weighted average**, where the weights are the model's own *attention* — its
guess of which slices matter (the weights sum to 1, so it really is an average):

```
slice:   1     2     3     4     5     6
Head A:  0.1   0.2   0.8   0.9   0.7   0.1     ← per-moment "how noisy?"
weight:  .05   .05   .30   .35   .20   .05     ← attention: which moments matter
                                                Head B = weighted avg ≈ 0.72  → "yes, noise"
```

Now the training. A Bronze clip only tells us *"a dog is in here somewhere"* → the correct Head B
answer is **1.0**. Suppose the model currently says Head B = 0.5, so training pushes it **up toward
1.0**. But Head B is that weighted average — **the only way to raise the average is to raise some of
the frame probabilities.** The gradients automatically lift the slices that had the **biggest
attention weights** (slices 3 & 4 here), i.e. the moments the model *already* leaned toward "noisy."
It leaves the low-weight, boring slices alone (lifting those barely moves the average).

> **Dimmer-switch analogy.** Picture 6 dimmer switches (the frame probabilities) and a meter on the
> wall showing their weighted-average brightness (Head B). Someone says *"make the meter read
> higher"* but **won't tell you which switch to touch.** The smart move: turn up the switches that
> already contribute most to the meter — the ones the model was already suspicious of. The dark,
> irrelevant switches stay off.

So a clip-level *yes/no* with **no timestamps** still nudges the *timeline* upward at the right
moments — never told *when*, yet the timing sharpens where it was already leaning. That's how even
the vaguest Bronze label still helps train the frame head.

### The "study buddy" trick (mean teacher)

We keep **two copies** of the model:
- a **student** (learns fast, but jumpy),
- a **teacher** (a slow, smoothed average of the student — steadier).

We show the student a slightly *messed-up* version of the audio (bits muted) and the teacher the
*clean* version, then demand: **"you two should still agree."** This forces the model to give stable
answers instead of flip-flopping — and it works on *all* tiers, even Bronze, because it needs no
labels, just consistency.

> Analogy: you and a calmer study partner check each other's answers. If you wildly disagree, one of
> you is guessing — so you both settle down.

### Why "two stages"?

WavLM (the ears) is already smart. If you let training rewrite it from the very first step, you'd
*scramble* everything it learned — like erasing an expert's memory to teach them one new word. So:

- **Stage 1:** *freeze* the ears (don't touch WavLM). Only train the new parts (BiGRU + heads) until
  they're decent.
- **Stage 2:** *gently* unfreeze the ears and fine-tune everything together with a **tiny** learning
  rate, so WavLM adjusts a little without forgetting.

---

## Putting it all together (one breath)

> Audio → **WavLM ears** describe each 20 ms → **BiGRU memory** adds context → **Frame head** says
> "noise now?" and **Clip head** says "noise anywhere?" → we train the Frame head with
> **Gold/Silver** (they have times) and the Clip head with **Bronze** (it doesn't) → a **study-buddy**
> keeps answers stable → do it in **two careful stages** so we don't wreck WavLM's ears → the frame
> timeline becomes your onset/offset predictions.
